### Notebook 05 — Phase A, Step A5: Synthetic Chat Generation
### Generate 3-turn customer support conversations using Gemini 2.5 Pro
### Strategy: Tiered template pools with dynamic financial number injection

In [1]:
import pandas as pd
import numpy as np
import json
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env", override=True)

# Load featured dataset
df = pd.read_csv("../data/processed/sbdr_featured_dataset.csv")
print(f"Dataset loaded: {df.shape}")
print(f"\nDistress distribution:")
print(df['distress_level'].value_counts().sort_index())

# Verify key columns exist for number injection
injection_cols = ['LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1', 'PAY_0', 
                  'lc_dti_mean', 'lc_annual_inc_mean', 'sp_avg_monthly_spend',
                  'spending_volatility', 'distress_level', 'customer_id']
missing = [c for c in injection_cols if c not in df.columns]
print(f"\nInjection columns missing: {missing if missing else 'None — all present'}")

# Configure OpenAI
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print(f"\nOpenAI key loaded: {os.environ['OPENAI_API_KEY'][:10]}...")
print("Client initialized")

Dataset loaded: (30000, 85)

Distress distribution:
distress_level
high         4500
low         10253
moderate    12614
severe       2633
Name: count, dtype: int64

Injection columns missing: None — all present

OpenAI key loaded: sk-proj-sB...
Client initialized


In [4]:
# Quick API test
test = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Return a JSON array with 2 objects, each having keys 'turn_1', 'turn_2', 'turn_3' with short placeholder text. Return ONLY valid JSON, no markdown."}],
    temperature=0.9
)

print("Status: SUCCESS")
print(f"\nRaw text:\n{test.choices[0].message.content}")

Status: SUCCESS

Raw text:
[
    {
        "turn_1": "Placeholder text for turn 1 of object 1.",
        "turn_2": "Placeholder text for turn 2 of object 1.",
        "turn_3": "Placeholder text for turn 3 of object 1."
    },
    {
        "turn_1": "Placeholder text for turn 1 of object 2.",
        "turn_2": "Placeholder text for turn 2 of object 2.",
        "turn_3": "Placeholder text for turn 3 of object 2."
    }
]


## Step 1: Define Conversation Generation Prompts
Each tier gets a system prompt that guides Gemini to produce realistic 3-turn chats.
Conversations include {placeholders} for financial numbers that get injected per-customer later.

In [5]:
# Prompt templates for each distress tier
# Each generates 3-turn conversations: Customer → Agent → Customer

tier_prompts = {
    "low": """You are generating synthetic customer support chat logs for a bank.
The customer has LOW financial distress — they are stable, paying on time, just has a routine inquiry.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): A routine question or minor request (balance inquiry, card replacement, transaction check)
- Turn 2 (Agent): Professional, helpful response
- Turn 3 (Customer): Satisfied, brief acknowledgment

Rules:
- Tone: Calm, polite, business-like
- Use these placeholders naturally in the customer's messages where relevant:
  {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}
- Keep each turn to 1-3 sentences
- Vary the topics: balance checks, card issues, payment confirmations, account settings
- Make each conversation distinct — different opening, different topic

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "moderate": """You are generating synthetic customer support chat logs for a bank.
The customer has MODERATE financial distress — they are starting to feel pressure, may have missed a payment, asking about options.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Mentions concern about upcoming payment, asks about flexibility, slight worry in tone
- Turn 2 (Agent): Reassuring, offers options like payment date change or temporary reduction
- Turn 3 (Customer): Relieved but still cautious, agrees to explore options

Rules:
- Tone: Slightly anxious but cooperative, not panicked
- Use these placeholders naturally: {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}, {{DTI}}
- Keep each turn to 1-3 sentences
- Topics: late payment concern, requesting extension, budget tightening, asking about fees
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "high": """You are generating synthetic customer support chat logs for a bank.
The customer has HIGH financial distress — they are struggling, may have lost income or faced a medical emergency, clearly stressed.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Reaches out about inability to pay, mentions hardship (job loss, medical bills, family emergency), emotional language
- Turn 2 (Agent): Empathetic, offers hardship program or restructuring options
- Turn 3 (Customer): Grateful but overwhelmed, expresses uncertainty about future, may ask follow-up about the program

Rules:
- Tone: Anxious, vulnerable, sometimes apologetic or frustrated
- Use these placeholders naturally: {{LIMIT_BAL}}, {{BILL_AMT}}, {{PAY_AMT}}, {{MONTHLY_SPEND}}, {{DTI}}, {{ANNUAL_INC}}
- Keep each turn to 2-4 sentences (these customers tend to explain more)
- Topics: job loss, medical emergency, divorce, reduced hours, unexpected expenses
- Show real emotional weight — this is someone in crisis reaching out for help
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
""",

    "severe": """You are generating synthetic customer support chat logs for a bank.
The customer has SEVERE financial distress — they may be completely unable to pay, emotionally shut down, avoidant, or in deep crisis.

Generate {batch_size} unique 3-turn conversations. Each conversation has:
- Turn 1 (Customer): Very short/avoidant OR emotionally distressed. May be responding to a bank outreach, not initiating. Mentions total inability to pay, or expresses hopelessness.
- Turn 2 (Agent): Very gentle, non-threatening, offers hardship restructuring or payment pause
- Turn 3 (Customer): Either minimal response ("ok", "I don't know"), emotional breakdown, or goes silent (just "..." or "I'll try")

Rules:
- Tone: Defeated, withdrawn, sometimes angry/defensive, sometimes silent
- Use these placeholders sparingly (these customers often don't reference specific numbers):
  {{BILL_AMT}}, {{PAY_AMT}}
- Keep customer turns SHORT (1-2 sentences max, sometimes just a few words)
- Topics: total payment stop, ignoring calls, feeling hopeless, mentions of depression/anxiety, anger at the system
- Some customers should be nearly non-communicative
- Make each conversation distinct

Return ONLY a valid JSON array. Each element:
{{"turn_1": "...", "turn_2": "...", "turn_3": "..."}}
"""
}

print("Tier prompts defined:")
for tier, prompt in tier_prompts.items():
    print(f"  {tier}: {len(prompt)} chars")

Tier prompts defined:
  low: 938 chars
  moderate: 979 chars
  high: 1176 chars
  severe: 1268 chars


## Step 2: Generate Conversation Pools via GPT-4o
200 unique conversations per tier × 4 tiers = 800 templates.
Batch size of 20 per API call = 40 calls total.

In [6]:
def generate_conversations(client, tier, prompt_template, batch_size=20, num_batches=10):
    """Generate conversation templates for a single distress tier."""
    all_conversations = []
    prompt = prompt_template.replace("{batch_size}", str(batch_size))
    
    for i in range(num_batches):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt + f"\n\nThis is batch {i+1} of {num_batches}. Make these completely different from any previous batch."}],
                temperature=1.0,
                max_tokens=4000
            )
            
            text = response.choices[0].message.content.strip()
            
            # Clean markdown code fences if present
            if text.startswith("```"):
                text = text.split("\n", 1)[1]
                text = text.rsplit("```", 1)[0]
                text = text.strip()
            
            conversations = json.loads(text)
            all_conversations.extend(conversations)
            print(f"  Batch {i+1}/{num_batches}: +{len(conversations)} convos (total: {len(all_conversations)})")
            
            time.sleep(1)
            
        except json.JSONDecodeError as e:
            print(f"  Batch {i+1}/{num_batches}: JSON parse error — {e}. Retrying...")
            time.sleep(3)
            try:
                response = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[{"role": "user", "content": prompt + f"\n\nBatch {i+1} retry. Return ONLY valid JSON array, no extra text."}],
                    temperature=0.9,
                    max_tokens=4000
                )
                text = response.choices[0].message.content.strip()
                if text.startswith("```"):
                    text = text.split("\n", 1)[1]
                    text = text.rsplit("```", 1)[0]
                    text = text.strip()
                conversations = json.loads(text)
                all_conversations.extend(conversations)
                print(f"  Batch {i+1}/{num_batches} (retry): +{len(conversations)} convos")
            except:
                print(f"  Batch {i+1}/{num_batches}: Retry failed. Skipping.")
            time.sleep(3)
        except Exception as e:
            print(f"  Batch {i+1}/{num_batches}: Error — {e}. Skipping.")
            time.sleep(5)
    
    return all_conversations

# Generate for all 4 tiers
conversation_pools = {}

for tier in ["low", "moderate", "high", "severe"]:
    print(f"\n{'='*50}")
    print(f"Generating {tier.upper()} tier conversations...")
    print(f"{'='*50}")
    
    conversations = generate_conversations(
        client=openai_client,
        tier=tier,
        prompt_template=tier_prompts[tier],
        batch_size=20,
        num_batches=10
    )
    conversation_pools[tier] = conversations
    print(f"\n{tier.upper()} complete: {len(conversations)} templates")

# Summary
print(f"\n{'='*50}")
print("GENERATION COMPLETE")
print(f"{'='*50}")
for tier, convos in conversation_pools.items():
    print(f"  {tier}: {len(convos)} templates")
print(f"  Total: {sum(len(c) for c in conversation_pools.values())} templates")

# Save raw pools as backup
with open("../data/processed/chat_pools_raw.json", "w") as f:
    json.dump(conversation_pools, f, indent=2)
print(f"\nRaw pools saved to data/processed/chat_pools_raw.json")


Generating LOW tier conversations...
  Batch 1/10: +20 convos (total: 20)
  Batch 2/10: +20 convos (total: 40)
  Batch 3/10: +20 convos (total: 60)
  Batch 4/10: +20 convos (total: 80)
  Batch 5/10: +20 convos (total: 100)
  Batch 6/10: +20 convos (total: 120)
  Batch 7/10: +20 convos (total: 140)
  Batch 8/10: +20 convos (total: 160)
  Batch 9/10: +20 convos (total: 180)
  Batch 10/10: +20 convos (total: 200)

LOW complete: 200 templates

Generating MODERATE tier conversations...
  Batch 1/10: +20 convos (total: 20)
  Batch 2/10: +20 convos (total: 40)
  Batch 3/10: +20 convos (total: 60)
  Batch 4/10: +20 convos (total: 80)
  Batch 5/10: +20 convos (total: 100)
  Batch 6/10: +20 convos (total: 120)
  Batch 7/10: +20 convos (total: 140)
  Batch 8/10: +20 convos (total: 160)
  Batch 9/10: +20 convos (total: 180)
  Batch 10/10: +20 convos (total: 200)

MODERATE complete: 200 templates

Generating HIGH tier conversations...
  Batch 1/10: +20 convos (total: 20)
  Batch 2/10: +20 convos (

## Step 3: Quality Check on Generated Pools
Inspect sample conversations from each tier to verify tone and placeholder usage.

In [7]:
# Sample 2 conversations from each tier
for tier in ["low", "moderate", "high", "severe"]:
    print(f"\n{'='*60}")
    print(f"  {tier.upper()} TIER — Sample Conversations")
    print(f"{'='*60}")
    
    samples = np.random.choice(len(conversation_pools[tier]), 2, replace=False)
    
    for idx in samples:
        convo = conversation_pools[tier][idx]
        print(f"\n--- Template #{idx} ---")
        print(f"  Customer: {convo['turn_1']}")
        print(f"  Agent:    {convo['turn_2']}")
        print(f"  Customer: {convo['turn_3']}")

# Check placeholder presence
print(f"\n{'='*60}")
print("PLACEHOLDER ANALYSIS")
print(f"{'='*60}")

placeholders = ["{{LIMIT_BAL}}", "{{BILL_AMT}}", "{{PAY_AMT}}", 
                "{{MONTHLY_SPEND}}", "{{DTI}}", "{{ANNUAL_INC}}"]

for tier, convos in conversation_pools.items():
    all_text = " ".join([c['turn_1'] + c['turn_2'] + c['turn_3'] for c in convos])
    counts = {p: all_text.count(p) for p in placeholders}
    active = {k: v for k, v in counts.items() if v > 0}
    print(f"  {tier}: {active if active else 'NO PLACEHOLDERS FOUND'}")


  LOW TIER — Sample Conversations

--- Template #102 ---
  Customer: Hi there, could you tell me if my statement for this month has been generated?
  Agent:    Yes, your monthly statement has been generated and is available to view online.
  Customer: Perfect, I'll check it out. Thanks!

--- Template #142 ---
  Customer: Could you verify if my last payment of {{PAY_AMT}} has been received?
  Agent:    I can confirm that your payment of $500 was successfully applied to your account.
  Customer: Perfect, thanks for confirming.

  MODERATE TIER — Sample Conversations

--- Template #2 ---
  Customer: With my current {{BILL_AMT}}, I'm feeling pressured financially. Are there any fee adjustments available?
  Agent:    We do offer ways to assist, such as waiving certain fees and providing a short-term payment relief option.
  Customer: That sounds promising. I'd like to know more about the fee waiving process.

--- Template #121 ---
  Customer: Hello, I've missed my last {{PAY_AMT}} and am c

## Step 4: Assign Conversations to Customers + Inject Real Financial Numbers
Each customer gets a random template from their tier's pool.
Placeholders get replaced with that customer's actual values from the dataset.

In [8]:
np.random.seed(42)

def inject_numbers(conversation, customer_row):
    """Replace placeholders with actual customer financial data."""
    replacements = {
        "{{LIMIT_BAL}}": f"${customer_row['LIMIT_BAL']:,.0f}",
        "{{BILL_AMT}}": f"${customer_row['BILL_AMT1']:,.0f}",
        "{{PAY_AMT}}": f"${customer_row['PAY_AMT1']:,.0f}",
        "{{MONTHLY_SPEND}}": f"${customer_row['sp_avg_monthly_spend']:,.0f}",
        "{{DTI}}": f"{customer_row['lc_dti_mean']:.1f}%",
        "{{ANNUAL_INC}}": f"${customer_row['lc_annual_inc_mean']:,.0f}"
    }
    
    result = {}
    for turn_key in ['turn_1', 'turn_2', 'turn_3']:
        text = conversation[turn_key]
        for placeholder, value in replacements.items():
            text = text.replace(placeholder, value)
        result[turn_key] = text
    
    return result

# Assign and inject
chat_records = []

for idx, row in df.iterrows():
    tier = row['distress_level']
    pool = conversation_pools[tier]
    
    # Pick a random template from the tier pool
    template = pool[np.random.randint(0, len(pool))]
    
    # Inject real numbers
    personalized = inject_numbers(template, row)
    
    chat_records.append({
        'customer_id': row['customer_id'],
        'distress_level': tier,
        'chat_turn_1': personalized['turn_1'],
        'chat_turn_2': personalized['turn_2'],
        'chat_turn_3': personalized['turn_3']
    })

chat_df = pd.DataFrame(chat_records)

print(f"Chat records created: {chat_df.shape}")
print(f"Null check: {chat_df.isnull().sum().sum()}")
print(f"\nDistribution:")
print(chat_df['distress_level'].value_counts().sort_index())

# Show a sample from each tier
for tier in ['low', 'moderate', 'high', 'severe']:
    sample = chat_df[chat_df['distress_level'] == tier].sample(1, random_state=42).iloc[0]
    print(f"\n{'='*60}")
    print(f"  {tier.upper()} — {sample['customer_id']}")
    print(f"{'='*60}")
    print(f"  Customer: {sample['chat_turn_1']}")
    print(f"  Agent:    {sample['chat_turn_2']}")
    print(f"  Customer: {sample['chat_turn_3']}")

Chat records created: (30000, 5)
Null check: 0

Distribution:
distress_level
high         4500
low         10253
moderate    12614
severe       2633
Name: count, dtype: int64

  LOW — SBDR_00351
  Customer: Hi, I need a replacement of my credit card that expires soon.
  Agent:    No problem, your new credit card is scheduled to be sent out before the expiration date of your current one.
  Customer: Thank you, that’s reassuring to know.

  MODERATE — SBDR_16504
  Customer: I missed a payment recently and I’m really feeling it. What can we do to manage this better?
  Agent:    We can review your account and see about adjusting your payment dates or amounts to make this easier.
  Customer: Thanks. I’d like to go over these adjustments with you.

  HIGH — SBDR_15218
  Customer: My partner just lost their job and it's critically impacted our finances. Managing $110,000 amidst the job loss is really distressing.
  Agent:    I'm really sorry to hear about the job loss. We can help by offering

## Step 5: Merge Chat Logs into Featured Dataset + Save Final A5 Output

In [9]:
# Merge on customer_id
df_final = df.merge(chat_df[['customer_id', 'chat_turn_1', 'chat_turn_2', 'chat_turn_3']], 
                     on='customer_id', how='left')

print(f"Final dataset shape: {df_final.shape}")
print(f"Null check: {df_final.isnull().sum().sum()}")
print(f"\nNew columns added: {[c for c in df_final.columns if c.startswith('chat_')]}")

# Save
df_final.to_csv("../data/processed/sbdr_final_dataset.csv", index=False)
print(f"\nSaved to data/processed/sbdr_final_dataset.csv")
print(f"File exists: {os.path.exists('../data/processed/sbdr_final_dataset.csv')}")

# Also save chat-only file for FinBERT training convenience
chat_df.to_csv("../data/processed/sbdr_chat_logs.csv", index=False)
print(f"Chat-only file saved to data/processed/sbdr_chat_logs.csv")

Final dataset shape: (30000, 88)
Null check: 0

New columns added: ['chat_turn_1', 'chat_turn_2', 'chat_turn_3']

Saved to data/processed/sbdr_final_dataset.csv
File exists: True
Chat-only file saved to data/processed/sbdr_chat_logs.csv


## Step 6: Final Verification — Complete Multi-Modal Customer Profile

In [10]:
# Show a complete customer from each tier with ALL data sources
for tier in ['low', 'moderate', 'high', 'severe']:
    sample = df_final[df_final['distress_level'] == tier].sample(1, random_state=42).iloc[0]
    print(f"\n{'='*60}")
    print(f"  COMPLETE PROFILE — {sample['customer_id']} ({tier.upper()})")
    print(f"{'='*60}")
    print(f"  Default: {sample['default payment next month']} | Avg Pay Delay: {sample['avg_pay_delay']:.2f}")
    print(f"  LC Profile: loan={sample['lc_loan_amnt_mean']:,.0f}, DTI={sample['lc_dti_mean']:.1f}%, income={sample['lc_annual_inc_mean']:,.0f}")
    print(f"  Sparkov: spend={sample['sp_total_spend']:,.0f}, volatility={sample['sp_spend_volatility']:.1f}")
    print(f"  PhraseBank: [{sample['pb_label']}] {sample['pb_sentence'][:80]}...")
    print(f"  Mental Health: [{sample['mh_status']}] {sample['mh_statement'][:80]}...")
    print(f"  Chat Turn 1: {sample['chat_turn_1'][:100]}...")
    print(f"  Chat Turn 2: {sample['chat_turn_2'][:100]}...")
    print(f"  Chat Turn 3: {sample['chat_turn_3'][:100]}...")

# Column count by source
print(f"\n{'='*60}")
print(f"  DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Total: {df_final.shape[0]} customers × {df_final.shape[1]} columns")
print(f"  UCI original: {len([c for c in df_final.columns if not c.startswith(('lc_','sp_','pb_','mh_','chat_','customer_id','distress_level','avg_pay_delay'))])} cols")
print(f"  Lending Club: {len([c for c in df_final.columns if c.startswith('lc_')])} cols")
print(f"  Sparkov: {len([c for c in df_final.columns if c.startswith('sp_')])} cols")
print(f"  PhraseBank: {len([c for c in df_final.columns if c.startswith('pb_')])} cols")
print(f"  Mental Health: {len([c for c in df_final.columns if c.startswith('mh_')])} cols")
print(f"  Chat logs: {len([c for c in df_final.columns if c.startswith('chat_')])} cols")
print(f"  Engineered: customer_id, distress_level, avg_pay_delay")
print(f"\n  Nulls: {df_final.isnull().sum().sum()}")


  COMPLETE PROFILE — SBDR_00351 (LOW)
  Default: 0 | Avg Pay Delay: -1.00
  LC Profile: loan=19,043, DTI=5.7%, income=132,919
  Sparkov: spend=52,209, volatility=121.6
  PhraseBank: [neutral] Runway Visual Range is a calculated assessment of the distance that a pilot can ...
  Mental Health: [Normal] tatiana k nope they didn t have it...
  Chat Turn 1: Hi, I need a replacement of my credit card that expires soon....
  Chat Turn 2: No problem, your new credit card is scheduled to be sent out before the expiration date of your curr...
  Chat Turn 3: Thank you, that’s reassuring to know....

  COMPLETE PROFILE — SBDR_16504 (MODERATE)
  Default: 0 | Avg Pay Delay: -0.17
  LC Profile: loan=18,985, DTI=18.5%, income=81,132
  Sparkov: spend=139,396, volatility=107.1
  PhraseBank: [neutral] Customers in a wide range of industries use our stainless steel and services wor...
  Mental Health: [Normal] I just tried taking a nap in my bed today. I've been sleeping on the couch since...
  Chat Turn

## ✅ Notebook 05 Complete — Phase A, Step A5

**What we did:**
- Generated 800 unique 3-turn conversation templates via GPT-4o (200 per tier)
- Tier-specific tone: low=calm, moderate=concerned, high=distressed, severe=withdrawn
- Injected real customer financial numbers into placeholders for personalization
- Assigned conversations to all 30K customers matched by distress level

**Final dataset:** `data/processed/sbdr_final_dataset.csv`
- **30,000 customers × 88 columns**
- Zero nulls
- Each customer now has: payment sequences + demographics + loan metrics + spending profile + financial text + emotional text + 3-turn chat log

**Phase A: COMPLETE** ✅
- A1 ✅ UCI exploration + feature engineering
- A2 ✅ Load remaining datasets
- A3 ✅ Merge all datasets via synthetic Customer ID
- A4 ✅ Full feature engineering (85 columns)
- A5 ✅ Synthetic chat generation (88 columns)

**Next → Phase B: Model Training**
- Branch 1: Fine-tune FinBERT on chat logs → Distress Score
- Branch 2: Train BiLSTM on payment sequences → Stress Pattern Vector
- Branch 3: Train XGBoost combining all signals → Recovery Tier Prediction